# Combined Image Retrieval & Patch Localization

Pipeline: **DINOv2 + multi-rotation + FAISS → SIFT/RANSAC re-ranking → SIFT/FLANN/RANSAC patch localization**.

This notebook was created from the supplied code, preserving its logic while organizing it into runnable notebook cells.

## Setup

In [ ]:
!pip install faiss-cpu opencv-contrib-python torch torchvision transformers matplotlib numpy --quiet

import os, glob
import cv2
import numpy as np
import torch
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Locate dataset images + query patch

In [ ]:
# ---------------------------------------------------
# 1. Locate dataset images + query patch already in /content
# ---------------------------------------------------
CONTENT_DIR = "/content"

all_jpgs = glob.glob(os.path.join(CONTENT_DIR, "*.jpg"))
dataset_paths = sorted([p for p in all_jpgs
                         if os.path.splitext(os.path.basename(p))[0].isdigit()])
print(f"Dataset images found: {len(dataset_paths)}")

query_candidates = glob.glob(os.path.join(CONTENT_DIR, "query_patch.*"))
assert len(query_candidates) > 0, f"Not found: query_patch.* in {CONTENT_DIR}"
QUERY_PATCH_PATH = query_candidates[0]
print(f"Query patch: {QUERY_PATCH_PATH}")

_img = Image.open(QUERY_PATCH_PATH); print("Query patch OK, size:", _img.size)
_img2 = Image.open(dataset_paths[0]); print("First dataset image OK, size:", _img2.size)

## Load DINOv2 backbone

In [ ]:
# ---------------------------------------------------
# 2. Load DINOv2 backbone
# ---------------------------------------------------
MODEL_NAME = "facebook/dinov2-base"
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

@torch.no_grad()
def get_embedding_from_pil(pil_img):
    """Global CLS embedding from DINOv2, L2-normalized."""
    inputs = processor(images=pil_img, return_tensors="pt").to(device)
    outputs = model(**inputs)
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    cls_embedding = torch.nn.functional.normalize(cls_embedding, p=2, dim=1)
    return cls_embedding.cpu().numpy().astype("float32")

def get_embedding(img_path):
    img = Image.open(img_path).convert("RGB")
    return get_embedding_from_pil(img)

## Multi-rotation indexing

In [ ]:
# ---------------------------------------------------
# 3. Multi-rotation indexing
#    Each dataset image is embedded at several rotation angles,
#    all vectors go into one FAISS index, mapped back to their
#    ORIGINAL source image.
# ---------------------------------------------------
ROTATION_ANGLES = [0, 60, 120, 180, 240, 300]  # 6 rotations per image

def rotate_pil(pil_img, angle):
    return pil_img.rotate(angle, expand=True, fillcolor=(255, 255, 255))

print(f"\nIndexing dataset with {len(ROTATION_ANGLES)} rotations per image...")
all_vectors = []
vector_to_image_idx = []

for img_idx, p in enumerate(dataset_paths):
    base_img = Image.open(p).convert("RGB")
    for angle in ROTATION_ANGLES:
        rotated = rotate_pil(base_img, angle) if angle != 0 else base_img
        emb = get_embedding_from_pil(rotated)
        all_vectors.append(emb)
        vector_to_image_idx.append(img_idx)
    if (img_idx + 1) % 20 == 0:
        print(f"  Indexed {img_idx + 1}/{len(dataset_paths)} images...")

all_vectors = np.vstack(all_vectors)
vector_to_image_idx = np.array(vector_to_image_idx)
print("Total indexed vectors:", all_vectors.shape)

dim = all_vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(all_vectors)

## DINO query + wide shortlist

In [ ]:
# ---------------------------------------------------
# 4. Query: search over ALL rotated vectors, get a WIDE shortlist
# ---------------------------------------------------
query_embedding = get_embedding(QUERY_PATCH_PATH)

SEARCH_K = 200  # search wide across all rotation variants
scores, indices = index.search(query_embedding, min(SEARCH_K, all_vectors.shape[0]))
scores, indices = scores[0], indices[0]

best_score_per_image = {}
for score, vec_idx in zip(scores, indices):
    img_idx = int(vector_to_image_idx[vec_idx])
    if img_idx not in best_score_per_image or score > best_score_per_image[img_idx]:
        best_score_per_image[img_idx] = score

ranked_images = sorted(best_score_per_image.items(), key=lambda x: -x[1])

SHORTLIST_SIZE = 20  # widen since DINO scores are unreliable when close together
shortlist = ranked_images[:SHORTLIST_SIZE]

print(f"\nDINO shortlist (top {SHORTLIST_SIZE}, before SIFT re-ranking):")
for rank, (img_idx, score) in enumerate(shortlist):
    print(f"  #{rank+1}: {os.path.basename(dataset_paths[img_idx])}  (dino sim = {score:.4f})")

fig, axes = plt.subplots(1, min(5, len(shortlist)), figsize=(4*min(5, len(shortlist)), 4))
if min(5, len(shortlist)) == 1:
    axes = [axes]
for ax, (img_idx, score) in zip(axes, shortlist[:5]):
    img = Image.open(dataset_paths[img_idx]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{os.path.basename(dataset_paths[img_idx])}\ndino sim={score:.3f}")
    ax.axis("off")
plt.suptitle("Task 1a: DINOv2 Multi-Rotation Shortlist (top 5 of 20 shown)")
plt.tight_layout()
plt.show()

## SIFT + RANSAC re-ranking

In [ ]:
# ---------------------------------------------------
# 5. Re-rank shortlist with SIFT+RANSAC inlier counting
#    This is the real decision-maker, not the DINO score.
# ---------------------------------------------------
def count_sift_inliers(query_path, candidate_path, resize_max=1200):
    def load_gray(path):
        img = cv2.imread(path)
        if img is None:
            return None
        h, w = img.shape[:2]
        scale = resize_max / max(h, w)
        if scale < 1:
            img = cv2.resize(img, (int(w*scale), int(h*scale)))
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    q_gray = load_gray(query_path)
    c_gray = load_gray(candidate_path)
    if q_gray is None or c_gray is None:
        return 0

    sift_rerank = cv2.SIFT_create(nfeatures=4000, contrastThreshold=0.02, edgeThreshold=15)
    kp_q, des_q = sift_rerank.detectAndCompute(q_gray, None)
    kp_c, des_c = sift_rerank.detectAndCompute(c_gray, None)

    if des_q is None or des_c is None or len(kp_q) < 4 or len(kp_c) < 4:
        return 0

    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

try:
        knn_matches = flann.knnMatch(des_q, des_c, k=2)
except cv2.error:
        return 0

    good = [m for m, n in knn_matches if len(knn_matches) > 0 and m.distance < 0.75 * n.distance]
    if len(good) < 4:
        return 0

    pts_q = np.float32([kp_q[m.queryIdx].pt for m in good])
    pts_c = np.float32([kp_c[m.trainIdx].pt for m in good])

    H, mask = cv2.findHomography(pts_q, pts_c, cv2.RANSAC, ransacReprojThreshold=5.0)
    if H is None:
        return 0

    return int(mask.sum())

print(f"\nRe-ranking shortlist with SIFT+RANSAC inlier counts...")
sift_scores = []
for img_idx, dino_score in shortlist:
    candidate_path = dataset_paths[img_idx]
    inliers = count_sift_inliers(QUERY_PATCH_PATH, candidate_path)
    sift_scores.append((img_idx, inliers, dino_score))
    print(f"  {os.path.basename(candidate_path)}: {inliers} inliers (dino sim={dino_score:.4f})")

sift_scores.sort(key=lambda x: -x[1])

print(f"\nFinal ranking (by SIFT inliers):")
for rank, (img_idx, inliers, dino_score) in enumerate(sift_scores[:5]):
    print(f"  #{rank+1}: {os.path.basename(dataset_paths[img_idx])}  ({inliers} inliers)")

fig, axes = plt.subplots(1, min(5, len(sift_scores)), figsize=(4*min(5, len(sift_scores)), 4))
if min(5, len(sift_scores)) == 1:
    axes = [axes]
for ax, (img_idx, inliers, dino_score) in zip(axes, sift_scores[:5]):
    img = Image.open(dataset_paths[img_idx]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{os.path.basename(dataset_paths[img_idx])}\n{inliers} inliers")
    ax.axis("off")
plt.suptitle("Task 1b: Re-ranked by SIFT inlier count (DINO shortlist -> SIFT verification)")
plt.tight_layout()
plt.show()

## Select best match

In [ ]:
# ---------------------------------------------------
# 6. Pick the best match by SIFT inliers -> feeds into Task 2 localization
# ---------------------------------------------------
best_idx, best_inliers, best_dino_score = sift_scores[0]
full_img_path = dataset_paths[best_idx]
query_img_path = QUERY_PATCH_PATH

print(f"\n>>> Best retrieved image: {full_img_path} ({best_inliers} SIFT inliers)")

if best_inliers < 8:
    print("⚠️ Warning: even the best candidate has few inliers — result may be unreliable. "
          "Consider widening SHORTLIST_SIZE or checking the query patch quality.")

print(">>> Proceeding to full-resolution SIFT+FLANN+RANSAC localization...\n")

## Task 2: SIFT patch localization

In [ ]:
# ===================================================
# Task 2: SIFT-based patch localization (full resolution)
# ===================================================

def load_and_resize(path, resize_max=1200):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not read image at: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    scale = resize_max / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (int(w*scale), int(h*scale)))
    return img

full_img_rgb = load_and_resize(full_img_path, resize_max=1200)
query_img_rgb = load_and_resize(query_img_path, resize_max=1200)

full_gray = cv2.cvtColor(full_img_rgb, cv2.COLOR_RGB2GRAY)
query_gray = cv2.cvtColor(query_img_rgb, cv2.COLOR_RGB2GRAY)

sift = cv2.SIFT_create(nfeatures=8000, contrastThreshold=0.02, edgeThreshold=15)
kp_query, des_query = sift.detectAndCompute(query_gray, None)
kp_full, des_full = sift.detectAndCompute(full_gray, None)

print(f"Query keypoints: {len(kp_query)}")
print(f"Full image keypoints: {len(kp_full)}")

if des_query is None or des_full is None or len(kp_query) < 4 or len(kp_full) < 4:
    raise RuntimeError("Not enough SIFT keypoints found. Try increasing resolution or "
                        "lowering contrastThreshold further (e.g. 0.01).")

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=100)
flann = cv2.FlannBasedMatcher(index_params, search_params)
knn_matches = flann.knnMatch(des_query, des_full, k=2)

good_matches = []
ratio_thresh = 0.75
for m, n in knn_matches:
    if m.distance < ratio_thresh * n.distance:
        good_matches.append(m)

print(f"Good matches after ratio test: {len(good_matches)} / {len(knn_matches)}")

if len(good_matches) < 4:
    raise RuntimeError("Too few good matches after ratio test. Try relaxing ratio_thresh "
                        "to 0.8-0.85, or increase nfeatures / lower contrastThreshold.")

mkpts0 = np.float32([kp_query[m.queryIdx].pt for m in good_matches])
mkpts1 = np.float32([kp_full[m.trainIdx].pt for m in good_matches])

H, inlier_mask = cv2.findHomography(mkpts0, mkpts1, cv2.RANSAC,
                                      ransacReprojThreshold=5.0,
                                      maxIters=5000, confidence=0.995)

if H is None:
    raise RuntimeError("Homography estimation failed — matches too inconsistent.")

num_inliers = int(inlier_mask.sum())
print(f"RANSAC inliers: {num_inliers} / {len(mkpts0)}")

if num_inliers < 8:
    print("⚠️ Warning: low inlier count — result may be unreliable. Consider relaxing "
          "ratio_thresh, increasing resolution, or verifying image content overlaps.")

## Visualize matches and localization

In [ ]:
def pad_to_height(img, target_h):
    h, w = img.shape[:2]
    pad = target_h - h
    return cv2.copyMakeBorder(img, 0, pad, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))

max_h = max(query_img_rgb.shape[0], full_img_rgb.shape[0])
q_padded = pad_to_height(query_img_rgb, max_h)
full_padded = pad_to_height(full_img_rgb, max_h)
combined = np.hstack([q_padded, full_padded])
offset = q_padded.shape[1]

fig, ax = plt.subplots(figsize=(18, 10))
ax.imshow(combined)
for i, ((x0, y0), (x1, y1)) in enumerate(zip(mkpts0, mkpts1)):
    color = 'lime' if inlier_mask[i] else 'red'
    ax.plot([x0, x1 + offset], [y0, y1], color=color, linewidth=0.6, alpha=0.6)
ax.axis('off')
plt.title(f"SIFT+FLANN: {num_inliers} inliers (green) / {len(mkpts0)} total (red=outlier)")
plt.show()

qh, qw = query_gray.shape
corners = np.array([[0, 0], [qw, 0], [qw, qh], [0, qh]], dtype=np.float32).reshape(-1, 1, 2)
projected_corners = cv2.perspectiveTransform(corners, H)

result_img = full_img_rgb.copy()
pts = projected_corners.reshape(-1, 2).astype(int)
cv2.polylines(result_img, [pts], isClosed=True, color=(255, 0, 0), thickness=4)

## Final result

In [ ]:
# ---------------------------------------------------
# 7. FINAL RESULT — retrieval + localization together
# ---------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(Image.open(query_img_path).convert("RGB"))
axes[0].set_title("Query Patch")
axes[0].axis("off")

axes[1].imshow(Image.open(full_img_path).convert("RGB"))
axes[1].set_title(f"Retrieved Image (SIFT inliers={best_inliers}, dino sim={best_dino_score:.3f})\n{os.path.basename(full_img_path)}")
axes[1].axis("off")

axes[2].imshow(result_img)
axes[2].set_title(f"Final Localization ({num_inliers} inliers)")
axes[2].axis("off")

plt.suptitle("End-to-End: DINO Shortlist -> SIFT Re-rank (Task 1) -> SIFT Localization (Task 2)", fontsize=14)
plt.tight_layout()
plt.show()

x_min, y_min = pts.min(axis=0)
x_max, y_max = pts.max(axis=0)
print(f"\n=== FINAL RESULT ===")
print(f"Retrieved image: {os.path.basename(full_img_path)} (SIFT inliers = {best_inliers}, DINO sim = {best_dino_score:.4f})")
print(f"Localized bbox in that image: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")